---
# Script to automate Forms in JSON
---

This work intends to create multiple forms automatically from a excel

Measure:

 - lenght / l / m
 - area / a / m²
 - volume / v / m³
 - weight / w / kg
 - tonnage / ton / tons
 - amount / u / uns
 - time / t / h
 - area / ha / ha
 - liters / lit / l

Types of special camps

 - number
 - string
 - textArea
 - jsonLogic
 - select
 - selectMultiple
 - boolean
 - timestamp
 - arrayOfObjects (not developed)

---            

In [1]:
# --- Lybries ---
import json as js
import pandas as pd
import math
from collections import defaultdict

In [2]:
# readind excel
name="Exemplo.xlsx"
contract = pd.read_excel(name,sheet_name='Contrato', na_values='')
#transport = pd.read_excel('vpt.xlsx',sheet_name='Transporte', na_values='')
links = pd.read_excel(name,sheet_name='Link_campo', na_values='')
fields = pd.read_excel(name,sheet_name='Campos', na_values='')
options = pd.read_excel(name,sheet_name='options', na_values='')
innerfields = pd.read_excel(name,sheet_name='innerfields', na_values='')

#Exibição de cabeçalho

# contract.head()
# transport.head()
fields.head()
# links.head()

,Campo,apiName,dataType,jsonLogic,displayIf,width,style,addTotal
0,Comprimento (m),length,float,NaN,NaN,12,"{""numFmt"": ""0.00""}",NaN
1,Largura (m),width,float,NaN,NaN,12,"{""numFmt"": ""0.00""}",NaN
2,Espessura (m),height,float,NaN,NaN,12,"{""numFmt"": ""0.00""}",NaN
3,Área (m²),area,jsonLogic,"{""if"":[{""missing"":[""formData.length"",""formData...",NaN,12,"{""numFmt"": ""0.00""}",x
4,Volume (m³),volume,jsonLogic,"{""if"":[{""missing"":[""formData.length"",""formData...",NaN,12,"{""numFmt"": ""0.00""}",x


In [3]:
# functions

def rd(lista):
    l = []
    for i in lista:
        if i not in l:
            l.append(i)
    l.sort()
    return l

In [4]:
# For each service
for row in contract.iterrows():
    
    # ---
    # Inicializating
    # ---
    
    data={} 
    data["id"] = 1
    data["name"] = str(row[1]['id'])
    data["fields"] = []
    data['groups'] = []
    data["displayName"] = str(row[1]['Classe'])
    data["measurementColumns"] = []

    i=1
    j=1
    o=1
    
    isMobile = [{"var": "isMobile"}, True] # For hide in app: "displayIf": {"!==": isMobile},
    
    # ---
    # Quantitative
    # ---
    
    members = []

    # ---
    # Length

    if row[1]["id_Quantitativo"] == "l":
        
        measure = {"var":"formData.length"}
        missing = ["formData.length"]
        iff = [{'missing': missing},0,measure]
        
        data["fields"].append({  
            "id": i,
            "apiName": "length",
            "dataType": "float",
            "displayName": "Comprimento (m)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "length",
            "style": {
                "numFmt": "0.00"
            },
            "width": 12,
            "addTotal": True,
            "header": "Comprimento"
        })
        # ---

    # ---
    # Area

    elif row[1]["id_Quantitativo"] == "a":
        
        mult = [{"var": "formData.length"},{"var": "formData.width"}]
        measure = {"*" : mult}
        missing = ["formData.length","formData.width"]
        iff = [{'missing': missing},0,measure]
        
        data["fields"].append({  
            "id": i,
            "apiName": "length",
            "dataType": "float",
            "displayName": "Comprimento (m)"
        })
        members.append(i)
        i+=1

        data["fields"].append({  
            "id": i,
            "apiName": "width",
            "dataType": "float",
            "displayName": "Largura (m)"
        })
        members.append(i)
        i+=1

        # Area
        
        data["fields"].append({  
            "id": i,
            "logic": {
                "if" : iff
            },
            "apiName": "area",
            "dataType": "jsonLogic",
            "displayName": "Área (m²)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "length",
            "style": {
                "numFmt": "0.00"
            },
            "width": 12,
            "header": "Comprimento (m)"
        })

        data["measurementColumns"].append({  
            "key": "width",
            "style": {
                "numFmt": "0.00"
            },
            "width": 12,
            "header": "Largura (m)"
        })

        # Area
        
        data["measurementColumns"].append({  
            "key": "area",
            "logic": {
                "if" : iff
            },
            "style": {
                "numFmt": "0.000"
            },
            "width": 12,
            "addTotal": True,
            "header": "Área (m²)"
        })
        # ---

    # ---
    # Volume

    elif row[1]["id_Quantitativo"] == "v": # Volume
        
        mult = [{"var": "formData.length"},{"var": "formData.width"},{"var": "formData.height"}]
        multa = [{"var": "formData.length"},{"var": "formData.width"}]
        measure = {"*" : mult}
        missing = ["formData.length","formData.width","formData.height"]
        missinga = ["formData.length","formData.width"]
        iff = [{'missing': missing},0,measure]
        iffa = [{'missing': missinga},0,{"*": multa}]
        
        data["fields"].append({  
            "id": i,
            "apiName": "length",
            "dataType": "float",
            "displayName": "Comprimento (m)"
        })
        members.append(i)
        i+=1

        data["fields"].append({  
            "id": i,
            "apiName": "width",
            "dataType": "float",
            "displayName": "Largura (m)"
        })
        members.append(i)
        i+=1

        data["fields"].append({  
            "id": i,
            "apiName": "height",
            "dataType": "float",
            "displayName": "Altura/Espessura (m)"
        })
        members.append(i)
        i+=1

        # Area

        data["fields"].append({  
            "id": i,
            "logic": {
                "if" : iffa
            },
            "apiName": "area",
            "dataType": "jsonLogic",
            "displayName": "Área (m²)"
        })
        members.append(i)
        i+=1

        # Volume

        data["fields"].append({  
            "id": i,
            "logic": {
                "if" : iff
            },
            "apiName": "volume",
            "dataType": "jsonLogic",
            "displayName": "Volume (m³)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "length",
            "style": {
                "numFmt": "0.00"
            },
            "width": 12,
            "header": "Comprimento (m)"
        })

        data["measurementColumns"].append({  
            "key": "width",
            "style": {
                "numFmt": "0.00"
            },
            "width": 12,
            "header": "Largura (m)"
        })

        data["measurementColumns"].append({  
            "key": "height",
            "style": {
                "numFmt": "0.00"
            },
            "width": 12,
            "header": "Altura/Espessura (m)"
        })

        # Area

        data["measurementColumns"].append({  
            "key": "area",
            "logic": {
                "if" : iffa
            },
            "style": {
                "numFmt": "0.000"
            },
            "width": 12,
            "addTotal": True,
            "header": "Área (m²)"
        })

        # Volume

        data["measurementColumns"].append({  
            "key": "volume",
            "logic": {
                "if" : iff
            },
            "style": {
                "numFmt": "0.000"
            },
            "width": 12,
            "addTotal": True,
            "header": "Volume (m³)"
        })
        # ---
        # ---
        
    # ---
    # Weight

    elif row[1]["id_Quantitativo"] == "w":
        
        measure = {"var":"formData.weight"}
        missing = ["formData.weight"]
        iff = [{'missing': missing},0,measure]
        
        data["fields"].append({  
            "id": i,
            "apiName": "weight",
            "dataType": "float",
            "displayName": "Peso (kg)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "weight",
            "style": {
                "numFmt": "0.000"
            },
            "width": 12,
            "addTotal": True,
            "header": "Peso (kg)"
        })
        # ---
        
    # ---
    # Tonnage

    elif row[1]["id_Quantitativo"] == "ton":
        
        measure = {"var":"formData.tonnage"}
        missing = ["formData.tonnage"]
        iff = [{'missing': missing},0,measure]
        
        data["fields"].append({  
            "id": i,
            "apiName": "tonnage",
            "dataType": "float",
            "displayName": "Peso (t)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "tonnage",
            "style": {
                "numFmt": "0.000"
            },
            "width": 12,
            "addTotal": True,
            "header": "Peso (t)"
        })
        # ---
        
    # ---
    # Amount

    elif row[1]["id_Quantitativo"] == "u":
        
        measure = {
            "var":"formData.amount"
        }
        
        missing = ["formData.amount"]
        
        iff = [{'missing': missing},0,measure]
        
        data["fields"].append({  
            "id": i,
            "apiName": "amount",
            "dataType": "float",
            "displayName": "Unidades (uns)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "amount",
            "style": {
                "numFmt": "0"
            },
            "width": 12,
            "addTotal": True,
            "header": "Quantidade (uns)"
        })
        # ---
        
    # ---
    # Area ah

    elif row[1]["id_Quantitativo"] == "ha":
        
        mult = [{"var": "formData.length"},{"var": "formData.width"},0.001]
        measure = {"*" : mult}
        missing = ["formData.length","formData.width"]
        iff = [{'missing': missing},0,measure]
        
        data["fields"].append({  
            "id": i,
            "apiName": "length",
            "dataType": "float",
            "displayName": "Comprimento (m)"
        })
        members.append(i)
        i+=1

        data["fields"].append({  
            "id": i,
            "apiName": "width",
            "dataType": "float",
            "displayName": "Largura (m)"
        })
        members.append(i)
        i+=1
        
        # Area ha

        data["fields"].append({  
            "id": i,
            "logic": {
                "if" : iff
            },
            "apiName": "area",
            "dataType": "jsonLogic",
            "displayName": "Área (m²)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "length",
            "style": {
                "numFmt": "0.00"
            },
            "width": 12,
            "header": "Comprimento (m)"
        })

        data["measurementColumns"].append({  
            "key": "width",
            "style": {
                "numFmt": "0.00"
            },
            "width": 12,
            "header": "Largura (m)"
        })

        data["measurementColumns"].append({  
            "key": "area",
            "logic": {
                "if" : iff
            },
            "style": {
                "numFmt": "0.000"
            },
            "width": 12,
            "addTotal": True,
            "header": "Área (ha)"
        })
        # ---
        
    # ---
    # Time

    elif row[1]["id_Quantitativo"] == "t":
        
        
        measure = {"var":"formData.time"}
        missing = ["formData.time"]
        iff = [{'missing': missing},0,measure]
        
        data["fields"].append({  
            "id": i,
            "apiName": "time",
            "dataType": "float",
            "displayName": "Tempo (h)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "time",
            "style": {
                "numFmt": "0.0"
            },
            "width": 12,
            "addTotal": True,
            "header": "Tempo (h)"
        })
        # ---
           
    # ---
    # Litros

    elif row[1]["id_Quantitativo"] == "lit":
        
        
        measure = {"var":"formData.volumeLiters"}
        missing = ["formData.volumeLiters"]
        iff = [{'missing': missing},0,measure]
        
        data["fields"].append({  
            "id": i,
            "apiName": "volumeLiters",
            "dataType": "float",
            "displayName": "Volume (l)"
        })
        members.append(i)
        i+=1

        data['groups'].append({
            'order': o,
            'members': members,
            'displayName': 'Quantitativos'
        })
        o+=1
        
        data["measurementColumns"].append({  
            "key": "volumeLiters",
            "style": {
                "numFmt": "0.0"
            },
            "width": 12,
            "addTotal": True,
            "header": "Volume (l)"
        })
        # ---
 
    # ---
    # DMT
    # ---
    
    # Alteração feita para a via paulista, nao atualizei essa parte
    # Colocar measurement e pegar informações pelo nome das colunas row[1][2] >> row[1]['lote']
    
    if row[1]["Transporte"] == 'x':
        
        transporte = transport.loc[transport['code'] == row[1][2]]
        
        for materials in transporte.iterrows():
            
            members = []
            
            manualDmt = [{"var": "data.formData.manualDmtInsert"+materials[1][6]},True]
            
            data["fields"].append({  
                "id": i,
                "apiName": "density"+materials[1][6],
                "logic": materials[1][7],
                "dataType": "jsonLogic",
                "displayName": "Fator de utilização (t/"+materials[1][12]+")"
            })
            members.append(i)
            i+=1
            
            if materials[1][11] == 1:
               
                moment = {"*":[materials[1][8], {'if': iff}]}
                momentWeight = {"*":[materials[1][8], materials[1][7], {'if': iff}]}
                
                data["fields"].append({
                    "id": i,
                    "logic": materials[1][8],
                    "apiName": "dmt"+materials[1][6],
                    "dataType": "jsonLogic",
                    "displayIf": {
                        "!==": manualDmt
                    },
                    "displayName": "DMT (km)"
                })
                members.append(i)
                i+=1
                
                data["fields"].append({
                    "id": i,
                    "logic": moment,
                    "apiName": "moment" + materials[1][6],
                    "dataType": "jsonLogic",
                    "displayIf": {
                        "!==": manualDmt
                    },
                    "displayName": "Momento de transporte ("+materials[1][12]+"*km)"
                })
                members.append(i)
                i+=1
                
                data["fields"].append({
                    "id": i,
                    "logic": momentWeight,
                    "apiName": "momentWeight" + materials[1][6],
                    "dataType": "jsonLogic",
                    "displayIf": {
                        "!==": manualDmt
                    },
                    "displayName": "Momento de transporte (t*km)"
                })
                members.append(i)
                i+=1
                # ---
                
            moment2 = {"*":[{'var':"dmt2" + materials[1][6]}, {'if': iff}]}
            momentWeight2 = {"*":[{'var':"dmt2" + materials[1][6]}, materials[1][7], {'if': iff}]}
                
            data["fields"].append({
                "id": i,
                "apiName": "manualDmtInsert" + materials[1][6],
                "dataType": "boolean",
                "displayName": "Inserir Outro valor para DMT"
            })
            members.append(i)
            i+=1
            
            data["fields"].append({
                "id": i,
                "apiName": "dmt2" + materials[1][6],
                "dataType": "float",
                "displayIf": {
                    "===": manualDmt
                },
                "displayName": "DMT (km)"
            })
            members.append(i)
            i+=1
            
            data["fields"].append({
                "id": i,
                "logic": moment2,
                "apiName": "moment2"+materials[1][6],
                "dataType": "jsonLogic",
                "displayIf": {
                    "===": manualDmt
                },
                "displayName": "Momento de transporte ("+materials[1][12]+"*km)"
            })
            members.append(i)
            i+=1
            
            data["fields"].append({
                "id": i,
                "logic": momentWeight2,
                "apiName": "momentWeight2"+materials[1][6],
                "dataType": "jsonLogic",
                "displayIf": {
                    "===": manualDmt
                },
                "displayName": "Momento de transporte (t*km)"
            })
            members.append(i)
            i+=1
                
            data['groups'].append({
                'order': o,
                "displayIf": {
                    "!==": isMobile
                },
                'members': members,
                'displayName': 'Transporte - ' + materials[1][5]
            })
            o+=1
            # ---
        # ---
    
    # ---
    # Especiais
    # ---
    
    members = []
    
    print("---")
    print("\n",row[1]['Classe'],"\n")
    print("---")
    
    if not pd.isnull(row[1]['Outros_campos']):
        
        campos = links.loc[links['id'] == row[1]["id"]].reset_index()
        listaGroup = rd(campos["group"])
        listaDeGroups=defaultdict(list)
        
        camn = 0
        
        for campo in campos.iterrows(): #string, textArea, number, float, boolean and timestamp
            print(campo[1]["Classe"],"-",campo[1]["Campo"])

            infos = fields.loc[fields['apiName'] == campo[1]["apiName"]].reset_index()

            c = {
                    "id": i,
                    "apiName": infos['apiName'][0],
                    "dataType": infos['dataType'][0],
                    "displayName": infos['Campo'][0]
                }

            m = {
                    "key": infos['apiName'][0],
                    "header": infos['Campo'][0]
                }

            if pd.notna(infos['width'][0]):
                m['width'] = infos['width'][0]
            if pd.notna(infos['style'][0]):
                m['style'] = eval(infos['style'][0])
            if pd.notna(infos['addTotal'][0]):
                m['addTotal'] = infos['addTotal'][0]
            if infos['dataType'][0] == "jsonLogic": #logic
                c["logic"] = eval(infos['jsonLogic'][0])
                m["logic"] = eval(infos['jsonLogic'][0])

            if infos['dataType'][0] == "select" or infos['dataType'][0] == "selectMultiple": #select and selectMultiple

                value = 1
                op = []
                catm = []
                listateste = pd.notna(options[infos['apiName'][0]])

                for index, opt in options[infos['apiName'][0]][listateste].iteritems():
                    op.append({
                        "name": opt,
                        "value": str(value)
                    })

                    ifm = []
                    inm = []
                    inm.append(str(value))
                    inm.append({
                        "var": "formData."+infos['apiName'][0]
                    })
                    ifm.append({"in": inm})
                    ifm.append(opt)
                    ifm.append("")
                    catm.append({"if": ifm})

                    value+=1  

                c["selectOptions"] = {
                    "options": op
                }

                m["logic"] = {"cat": catm}

            if not pd.isnull(infos["displayIf"][0]): #displayIf
                c["displayIf"] = eval(infos["displayIf"][0])

            data['fields'].append(c)
            data['measurementColumns'].append({**m, "width": int(m['width'])})

            listaDeGroups[campos["group"][camn]].append(i)
            i+=1
            camn+=1

        # ---
        # ADIÇÃO DE GRUPOSS!!!!!!!!
        # ---
                
        for x in listaGroup: 
            data['groups'].append({
                'order': o,
                'members': listaDeGroups[x],
                'displayName': str(x)
            })
            o+=1
            
            print("\n Grupo ",x," foi adicionado \n      Contém os campos",listaDeGroups[x])
            
    # ---
    # Notes
    # ---
    
    members=[]
    
    data['fields'].append({
        "id": i,
        "apiName": "notes",
        "dataType": "textArea",
        "displayName": "Observações"
    })
    members.append(i)
    i+=1
    
    data['groups'].append({
        'order': o,
        'members': members,
        'displayName': 'Informações adicionais'
    })
    o+=1
    
    data["measurementColumns"].append({
        "key": "notes",
        "logic": {
            "var": "formData.notes"
        },
        "width": 20,
        "header": "Observa\u00e7\u00f5es"
    })
    
    # ---
    # EXPORT
    # ---
    
    with open(str(row[1]["id"])+'.txt', 'w') as outfile:  
        js.dump(data, outfile, indent=4)
        
        

---

 Remendo profundo 

---
---

 Fresagem e recomposição 

---
---

 Microrrevestimento 

---
---

 Tapa Buraco 

---
---

 Dreno Longitudinal 15x60 

---
---

 Dreno Transversal 13x30 

---
---

 Cerca 4 Fios 

---
---

 Faixa de sinalização Definitiva 

---
---

 Tachão refletivo bidirecional 

---
---

 Placa 

---
---

 Dreno 

---
---

 Limpeza de Placa 

---
